### Git repository:
https://github.com/KeremOzemre/Computational-Social-Science.git

#### Main areas of contributions:

##### Part 1: s244322, s245290
##### Part 2: s244794, s245290
##### Part 3: S244794, s244322

# Part 1

#### 1) What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)?
Centola’s study uses custom-made data because the researcher designed an online experiment with controlled network structures. The main advantage was that researchers could control the study conditions, which makes it easier to identify what causes what, reduce the influence of other factors, and understand how social influence (like reinforcement from others) affects behaviour. This fits the strengths of experiments: like high internal validity and clear controlled conditions. However, custom-made data are costly, involve smaller samples, and may create artificial settings that reduce real-world realism.

Nicolaides’ study uses ready-made data, analysing large-scale fitness-tracking and social network traces. Advantages include huge samples, real-world behaviour, and long time scales, which makes the results more representative of real-world behaviour and allows researchers to detect patterns more reliably because of the large amount of data. But, as discussed in Chapter 2.3, ready-made data can be incomplete, biased, dirty, and affected by confounding factors, making causal claims harder.

#### 2) How do you think these differences can influence the interpretation of the results in each study?
The differences between custom-made and ready-made data influence how the results should be interpreted. In Centola’s experiment the controlled design makes it easier to interpret the results as causal, meaning that differences in behaviour can more confidently be explained by the network structure itself. However, because the setting was artificial and more limited, the findings might not fully show how behavior spreads in real-world social networks. In contrast, Nicolaides’ study uses real-world data from a large population, which makes the results more realistic and broadly aplicable. However, because the data were not created for research purposes, there may be hidden biases and confounding factors meaning the results should be interpreted more as strong associations rather than clear causal effects. Overall for the interpretation, Centola provides stronger causal evidence but lower realism, while Nicolaides offers higher realism and scale but more uncertainty about causality.



# Part 2

### Importing packages

In [1]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
from itertools import islice
from joblib import Parallel, delayed
from tqdm import tqdm
import ast
API_KEY = os.getenv("API_KEY")
authors_file = open('authors.txt','r')

### Extracting Open Alex by calling the API on names extracted from Web Scraping

In [5]:
import requests
import pandas as pd
import time

rows = []
batch_size = 25 # number of authors per batch
sleep_time = 1   # seconds to wait between batches

authors = [a.strip() for a in authors_file if a.strip()]
total = len(authors)

for start in range(0, total, batch_size):
    batch = authors[start:start + batch_size]
    for author in batch:
        URL = "https://api.openalex.org/authors"
        params = {
            "search": author,
            "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
            "api_key": API_KEY
        }
        try:
            response = requests.get(URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {author}: {e}")
            continue

        if data.get("results"):
            author_data = data["results"][0]
            h_index = author_data.get("summary_stats", {}).get("h_index")
            institutions = author_data.get("last_known_institutions", [])
            country_code = institutions[0].get("country_code") if institutions else None

            rows.append({
                "searched_name": author,
                "display_name": author_data.get("display_name"),
                "works_count": author_data.get("works_count"),
                "author_id": author_data.get("id"),
                "works_api_url": author_data.get("works_api_url"),
                "h_index": h_index,
                "country_code": country_code
            })
        else:
            print(f"No results found for author: {author}")

    print(f"Processed batch {start // batch_size + 1} / {total // batch_size + 1}")
    time.sleep(sleep_time)  # pause between batches to avoid throttling

df = pd.DataFrame(rows)

No results found for author: Adam (Zhengzi) Zhou
No results found for author: Amirhossein Nakhaei
Processed batch 1 / 64
No results found for author: Ic2S2 Program Chairs
No results found for author: Vincent Christoph Brockers
Processed batch 2 / 64
No results found for author: Justyna Janczy
Processed batch 3 / 64
Processed batch 4 / 64
Processed batch 5 / 64
Processed batch 6 / 64
Processed batch 7 / 64
No results found for author: Markus Bo Pettersson
No results found for author: Jesica Olivares
Processed batch 8 / 64
Processed batch 9 / 64
No results found for author: Kareem Elrafei
No results found for author: Sandrine Lara Chausson
Processed batch 10 / 64
No results found for author: Dominik Loibner
No results found for author: Alejandro De La Fuente Cuesta
No results found for author: Pamina Syed Al
Processed batch 11 / 64
No results found for author: Adolfo Fuentes-Jofre
No results found for author: Cristian E Candia
Processed batch 12 / 64
No results found for author: Laura Ma

### Reflection questions for part 2:

#### Which challenges did you encounter? How did you address them?

Firstly, by looping over each individual, we realized that the process was taking a very long time to complete. Hence, we decided to batch the display names in groups of 25 (the maximum permitted by the OpenAlex API—to improve runtime efficiency). Furthermore, batching allowed us to experiment further without exceeding the API request limit granted under the free account, which proved to be another challenge during the initial stages of working with the API. In addition, we encountered issues related to exceeding the API call rate per second. This was solved through the use of batching and the implementation of a sleep function to pause execution between successive batch requests. Another challenge concerned data verification. At the outset, we struggled to determine whether missing data resulted from API errors, incorrect code implementation, or the absence of the author in the database. This issue was addressed by incorporating debugging print statements and applying controlled pauses to minimize potential API-related errors.

#### Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data?

The most significant problem arose after implementing batching, when we received the HTTP 429 “Too Many Requests” error. Digging deeper into this, we learned that this was caused by exceeding the predefined OpenAlex API rate limit per second. To address this, we introduced a one-second pause between batches to prevent further rate-limit violations. This approach also ensured that no batches were unintentionally skipped due to request errors. Given the size of the dataset, it was otherwise difficult to determine whether missing authors were the result of system errors or their absence from the database. Although this solution increased the total extraction time by approximately one minute, given that we processed 64 batches, we prioritized data integrity and completeness over execution speed.

# Part 3

### Filtering Authors by work count

In [6]:
indexes = df.index[
    (df["works_count"] >= 5) &
    (df["works_count"] <= 5000)
].tolist()

filtered_author_list = df.loc[indexes, "author_id"]

### Calling API for all works of all IC2S2 authors

In [7]:
# Concept filter ensuring only works within the intersection of computational social science and quantitative disciplines
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
quant_concepts = ["C33923547", "C121332964", "C41008148"]

concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

# send requests in chunks (25 max permitted by open alex) to filter multiple authors in the same request for faster extraction
def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

# Created function, instead of for loop as we can call the same function in parallel to again fasten the process
def process_single_batch(author_batch, batch_num):
    
    rows2_local = []
    rows3_local = []
    
    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
        f"{author_filter},"
        f"cited_by_count:>10,"
        f"authors_count:<10,"
        f"{concept_filter1},"
        f"{concept_filter2}"
    )
    
    cursor = "*"
    
    while cursor:
        params = {
            "filter": filter_string,
            "select": "id,publication_year,cited_by_count,title,abstract_inverted_index,authorships",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error in batch {batch_num}: {e}")
            break
        
        results = data.get("results", [])
        if not results:
            break
        
        for item in results:
            work_id = item.get("id")
            
            # find author id
            author_ids = [
                auth["author"]["id"]
                for auth in item.get("authorships", [])
                if auth.get("author") and auth["author"].get("id")
            ]
        
            
            rows2_local.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "author_ids": author_ids,
            })
            
            rows3_local.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })
        
        cursor = data["meta"].get("next_cursor")
        sleep(0.05)  # Small delay between pages to avoid going over the rate limit 
    
    return rows2_local, rows3_local

# Create batches
author_batches = list(chunked(filtered_author_list, 25))

# Process batches in parallel with tqdm progress bar
results = Parallel(n_jobs=4, verbose=0)(
    delayed(process_single_batch)(batch, i+1) 
    for i, batch in enumerate(tqdm(author_batches, desc="Processing batches"))
)

# Combine results
rows2 = []
rows3 = []
for batch_rows2, batch_rows3 in results:
    rows2.extend(batch_rows2)
    rows3.extend(batch_rows3)

# Create DataFrames
df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print(f"\nComplete!")
print(f"Works retrieved: {len(df2)}")
print(f"Abstracts retrieved: {len(df3)}")

Processing batches: 100%|██████████| 54/54 [00:24<00:00,  2.22it/s]



Complete!
Works retrieved: 12772
Abstracts retrieved: 12772


In [ ]:
df2.to_csv("IC2S2_papers.csv", index=False)
df3.to_csv("IC2S2_abstracts.csv", index=False)

#### Finding all unique authors from IC2S2 papers

In [13]:
work_author_ids = df2["author_ids"]
single_list_work_author_ids = [author for sublist in work_author_ids for author in sublist]
single_list_work_author_ids = set(single_list_work_author_ids)
len(single_list_work_author_ids)

18933

## Data Overview and Reflection questions:

### Dataset summary)

???

### Efficiency in code)
#### Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?

We filtered the authors based on their work count before passing the list to the API. Moreover, instead of calling all the works of all the filtered authors, we filtered during the works during the call, hence allowing for far less API calls to be made. Furthermore, we utilized Open Alex API's ability to parse up to 25 requests at once and called for 25 distinct authors at once instead of one by one. Finally, we used joblib's parallel function to deploy 4 parallel batched API calls, significantly reducing the extraction time. With all these efficiency techniques, we were able to extract both the IC2S2 works and IC2S2 abstracts datasets in around 30 seconds. During employing these techniques we payed attention to not exceeding the rate limit we implemented pauses by sleep to ensure to not miss any authors by overworking the API. 

### Filtering Criteria and Dataset Relevance)
#### Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?

The rationale behind the applied filters, were firstly to keep the dataset to a maintable size which relates more to the work counts upper limit and authors count limit. The concept, cited by count and the lower work count filters, were applied to ensure the relevance of the works as this enabled only works with a certain relevance to other researchers as well as only more established auhthors getting considered. Although, relevance of works were ensured, the filters also allowed for a bias in the dataset. Only established authors' work's were considered having an underrepresentation of the emerging works, furthemore works which focused further on the social or the computational aspect a certain topic were discriminated by the concept's filter. The upper work_count limit also lead to the exclusion of possibly very crucial authors, and hence their works which may be essential to computational social science.